In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import ast
import pandas as pd
import numpy as np
from src.config import *

In [2]:
# 1. CARGA Y RENOMBRE COMPLETO (Recuperado de tu código original)
# ==============================================================================
df_raw = pd.read_excel("../data/pacientes.xlsx")
hospitales = pd.read_csv("../data/hospitales_coordenadas.csv")

dict_comp = dict(zip(hospitales['Nombre Hospital'], hospitales['complejidad']))
hospitales['color_rgb'] = hospitales['color'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df_base = df_raw.rename(columns={
    'Id Hospital': 'hospital_id', 'Nombre Hospital': 'hospital_origen',
    'Id': 'paciente_id', 'Fecha inicio': 'fecha_ingreso', 'Fecha egreso': 'fecha_egreso',
    'Estado al ingreso': 'estado_ingreso', 'Tipo al ingreso': 'tipo_ingreso',
    'Último estado': 'estado_ultimo', 'Último tipo': 'tipo_ultimo',
    'Sexo': 'sexo', 'Edad': 'edad', 'Nivel riesgo clínico': 'riesgo_clinico',
    'Nivel riesgo social': 'riesgo_social', 'Enfermedades preexistentes Covid-19': 'comorbilidades_covid',
    'Enfermedades preexistentes pediatría': 'comorbilidades_pediatria', 'Vacuna': 'vacuna',
    'Cant. dosis': 'cantidad_dosis', '1º dosis': 'fecha_dosis_1', '2º dosis': 'fecha_dosis_2',
    'Buscado en el ministerio': 'validado_ministerio', 'Obra social': 'obra_social',
    'Asistencia Respiratoria Mecánica': 'requiere_arm', 'Motivo': 'motivo_egreso',
    'Operación': 'operacion', 'Última actualización': 'fecha_ultima_actualizacion',
    'Pasó por Críticas': 'paso_criticas', 'Pasó por Intermedias': 'paso_intermedias',
    'Pasó por Generales': 'paso_generales'
}).copy()

df_base['hospital_origen'] = df_base['hospital_origen'].replace({
    'Módulo Hospitalario 11- FV': 'Módulo Hospitalario 11 - FV',
    'Módulo Hospitalario  9 - AB': 'Módulo Hospitalario 9 - AB'
}).str.strip()


df_base['fecha_ingreso'] = pd.to_datetime(df_base['fecha_ingreso'], errors='coerce')
df_base['fecha_egreso'] = pd.to_datetime(df_base['fecha_egreso'], errors='coerce')
df_base['edad'] = pd.to_numeric(df_base['edad'], errors='coerce')
df_base['dias_en_nodo'] = np.ceil((df_base['fecha_egreso'] - df_base['fecha_ingreso']).dt.total_seconds() / 86400).fillna(0).astype('Int64')

df_base = df_base.sort_values(['paciente_id', 'fecha_ingreso'])

In [3]:
# 2. FILTROS DE CALIDAD Y DESENLACE ROBUSTO (La nueva lógica)
# ==============================================================================

# Filtro Edad: que no pasen 2 anios de diferencia
diff_edad = df_base.groupby('paciente_id')['edad'].agg(lambda x: x.max() - x.min() if x.notna().any() else 0)
pacientes_edad_ok = diff_edad[diff_edad.fillna(0) <= TOLERANCIA_EDAD_ANIOS].index

# Desenlaces validos: que no tengan un final de caso que no corresponda
def obtener_fin_caso(g):
    motivos = g['motivo_egreso'].fillna('').str.lower()
    g['prioridad'] = motivos.map(ORDEN_PRIORIDAD_CLINICA).fillna(99)
    return g.sort_values(['prioridad', 'fecha_egreso'], ascending=[True, False]).iloc[0]['motivo_egreso']

desenlaces = df_base.groupby('paciente_id').apply(obtener_fin_caso).reset_index(name='motivo_final')
pacientes_final_ok = desenlaces[~desenlaces['motivo_final'].isin(MOTIVOS_A_DESCARTAR)].paciente_id

# Base Limpia Final (Episodios) : juntamos ambas cosas
lista_validos = set(pacientes_edad_ok).intersection(set(pacientes_final_ok))
df_base_limpia = df_base[df_base['paciente_id'].isin(lista_validos)].copy()
df_base_limpia['nivel_complejidad'] = df_base_limpia['hospital_origen'].map(dict_comp).fillna('Desc').astype(str).str.replace('.0', '', regex=False)

In [ ]:
# 3. TRASLADOS Y TRAYECTORIAS (Validación Cruzada y Tolerancia Administrativa)
# ==============================================================================

# A. Preparación de variables temporales y de destino (El "shift")
df_base_limpia['hosp_destino_real'] = df_base_limpia.groupby('paciente_id')['hospital_origen'].shift(-1)
df_base_limpia['fecha_ingreso_destino'] = df_base_limpia.groupby('paciente_id')['fecha_ingreso'].shift(-1)

# Calculamos el "Gap" (Diferencia de días entre salir de A y entrar a B)
df_base_limpia['dias_traslado'] = (df_base_limpia['fecha_ingreso_destino'] - df_base_limpia['fecha_egreso']).dt.days

# Corrección técnica para traslados que ocurren el mismo día (horas de diferencia)
df_base_limpia.loc[df_base_limpia['dias_traslado'] == -1, 'dias_traslado'] = 0

# B. Preparación de variable de intención (Lo que anotó el médico)
df_base_limpia['medico_indico_traslado'] = df_base_limpia['motivo_egreso'].str.contains('traslado', na=False, case=False)


# ------------------------------------------------------------------------------
# MÁSCARA MAESTRA DE TRASLADOS: "Laxo en la administración, estricto en la clínica"
# ------------------------------------------------------------------------------
# CRITERIO A DISCUTIR: ¿Cómo definimos matemáticamente que dos internaciones son consecutivas?
# Decisión: Permitimos un "solapamiento" negativo de días para absorber demoras en 
# la carga del alta en el Hospital A, pero exigimos que la física del ingreso se respete.

mask_traslado = (
    # 1. REGLAS ESTRUCTURALES (Debe existir un salto real)
    (df_base_limpia['hosp_destino_real'].notna()) & 
    (df_base_limpia['hospital_origen'] != df_base_limpia['hosp_destino_real']) & 
    
    # 2. REGLAS TEMPORALES (La ventana de -5 a +5 días)
    # 2.a - Regla de Física: No puede ingresar a B ANTES de haber ingresado a A.
    (df_base_limpia['fecha_ingreso_destino'] >= df_base_limpia['fecha_ingreso']) &
    
    # 2.b - Regla del Hueco Máximo: Si sale de A, lo esperamos máximo 5 días en B.
    (df_base_limpia['dias_traslado'] <= DIAS_VENTANA_TRASLADO) & 
    
    # 2.c - Regla de Tolerancia Administrativa: Puede ingresar a B HASTA 5 días antes 
    # de que le firmen el alta oficial en A (solapamiento negativo por error de carga).
    (df_base_limpia['dias_traslado'] >= -DIAS_VENTANA_TRASLADO) & 
    
    # 3. REGLA DE INTENCIÓN MÉDICA (Validación cruzada)
    # esto quiero sacarlo:
    (
        df_base_limpia['medico_indico_traslado'] |                    # O bien dice explícitamente "traslado"
        df_base_limpia['motivo_egreso'].isin(['otro', 'anulado', '']) # O bien fue desprolijo y lo dejó ambiguo (lo salva la regla de tiempo)
    )

    # 
)

# Aplicamos la máscara para quedarnos solo con las "flechas" reales de la red
df_traslados = df_base_limpia[mask_traslado].copy()

# C. Consolidación de la base de Pacientes (1 fila por persona para rankings)
df_pacientes_trayectorias = df_base_limpia.groupby('paciente_id').agg(
    ruta_hospitales_str=('hospital_origen', lambda x: ' -> '.join(x.astype(str))),
    ruta_complejidad_str=('nivel_complejidad', lambda x: ' -> '.join(x.astype(str))),
    cantidad_traslados=('hospital_origen', lambda x: len(x) - 1),
    dias_estadia_total=('dias_en_nodo', 'sum'),
    fecha_ingreso_red=('fecha_ingreso', 'first'),
    fecha_egreso_red=('fecha_egreso', 'last')
).reset_index()

# Pegamos el desenlace estandarizado (Calculado previamente con la jerarquía clínica)
df_pacientes_trayectorias = df_pacientes_trayectorias.merge(desenlaces, on='paciente_id')
df_pacientes_trayectorias['motivo_fin_caso'] = df_pacientes_trayectorias['motivo_final'].map(MAPEO_NOMBRES_FINALES)

In [6]:
# 4. SANITY CHECK FINAL (NÚMEROS PARA EL MD)
# ==============================================================================
print("\n" + "="*40)
print("SANITY CHECK DE CALIDAD FINAL")
print("="*40)
print(f"1. Pacientes Originales: {df_base['paciente_id'].nunique()}")
print(f"2. Pacientes Descartados (Edad/Fin Inconcluso): {df_base['paciente_id'].nunique() - len(lista_validos)}")
print(f"3. Cohorte Final de Análisis: {len(df_pacientes_trayectorias)}")
print(f"4. Traslados Válidos Detectados: {len(df_traslados)}")
print("-" * 40)
print("Distribución de desenlaces finales:")
print(df_pacientes_trayectorias['motivo_fin_caso'].value_counts())


SANITY CHECK DE CALIDAD FINAL
1. Pacientes Originales: 27295
2. Pacientes Descartados (Edad/Fin Inconcluso): 4144
3. Cohorte Final de Análisis: 23151
4. Traslados Válidos Detectados: 1499
----------------------------------------
Distribución de desenlaces finales:
motivo_fin_caso
Alta Médica         17478
Defunción            2784
Hospital Externo     2589
Alta Hotel            300
Name: count, dtype: int64


KeyError: "['tipo_egreso'] not in index"